# **US Flights Delay:** Schema Engineering

## Imports

In [1]:
import sys

In [2]:
sys.path.append("../src/database")
sys.path.append("../src/database/queries")

In [3]:
import connection 
import base_queries

## Connect to client and database

In [4]:
MONGODB_URI = "mongodb://localhost:27017/"
MONGODB_NAME = "flights_delay_db"

In [5]:
client, database = connection.connect_to_database(uri = MONGODB_URI, db_name = MONGODB_NAME)

In [6]:
database.list_collection_names()

['airports',
 'airports_summary_hybrid_optimized',
 'runways',
 'cancelled_deverted_2023',
 'flights_hybrid_optimized',
 'us_flights_2023',
 'weather_hybrid_optimized',
 'airport_frequencies',
 'airports_geolocation',
 'us_flights_optimized',
 'weather_meteo_by_airport']

## Queries

### Flight Delay Analysis

**Question 1:** Which US states have the highest average delays by season (Winter, Spring, Summer, Fall)?

* Data from the **`us_flights_2023`** and **`airports_geolocation`** collections are analyzed.
* The season is determined based on the `flight_date` field:

  * **Winter:** December–February
  * **Spring:** March–May
  * **Summer:** June–August
  * **Fall:** September–November
* The average delay is calculated as `avg(dep_delay)` for all flights from a given state (`state` from `airports_geolocation`) in a given season.
* Result: List of US states with average delay by season, sorted in descending order of average delay.

---

**Question 2:** Which airlines have the highest average delays on rainy days?

* Collections **`us_flights_2023`** and **`weather_meteo_by_airport`** are used.
* Precipitation is taken from the `prcp' field (mm).
* **Significant precipitation**: days when `prcp > 5.0`.
* Need to find average delay (`avg(dep_delay)`) by airline (`airline`) only for days with significant precipitation, based on weather conditions from `departure.airport_code`.
* Result: airlines with average delay on days with precipitation > 5 mm, sorted in descending order of value.

---

**Question 3:** Which airports have the most canceled flights during bad weather?
* Collections **`cancelled_deverted_2023`**, **`weather_meteo_by_airport`** and **`airports_geolocation`** are used.
* Canceled flights are those with `cancelled = 1`.
* **Bad weather conditions** are defined as:

  * `prcp > 10 mm' *(heavy precipitation)*
  * **or** `wspd > 15 m/s' *(strong wind)*
* The query should match ($lookup) flights and weather data by `dep_airport` and `airport_id`.
* Result: airports with the highest number of canceled flights during bad weather, sorted in descending order of cancellations.

---

**Question 4:** Do airports with more diverse runways have lower average delays?

* **`airports`**, **`runways`**, and **`us_flights_2023`** collections are used.
* For each airport, the following is calculated:

  * **Runway Diversity Index (RDI)** = number of different values ​​of `surface` from the collection of `runways` per airport.
  * **Average Delay** = average `dep_delay` from `us_flights_2023` per airport.
* It is necessary to merge (`$lookup') all three collections, calculate both metrics, and analyze whether airports with higher RDI have lower average delays.
* Result: list of airports with RDI and average delay, sorted in ascending order of average delay.

---

**Question 5:** How does airline performance differ by type of flight distance (Short, Medium, Long Haul)?

* Collection **`us_flights_2023`** is used.
* `distance_type' indicates the flight length category.
* For each airline (`airline`) the average delay (`avg(arr_delay)`) is calculated, grouped by `distance_type`.
* Result: a table with airlines and average delay by route type, sorted descending by average delay within each category.

In [7]:
base_queries = base_queries.get_base_queries()

In [8]:
collection = database["us_flights_2023"]

**Q1:** Which US states have the highest average delays by season (Winter, Spring, Summer, Fall)?

In [9]:
base_query_1 = base_queries['query_1']

In [10]:
results = collection.aggregate(base_query_1)
results = list(results)

for i, docs in enumerate(results[:10], 1):
    print(f"{i}. \n{docs}\n\n")

In [11]:
debug_pipeline = [
    {
        "$match": {
            "Dep_Airport": {"$ne": None},
            "Dep_Delay": {"$ne": None},
            "FlightDate": {"$ne": None}
        }
    },
    {
        "$limit": 5  # Samo prvih 5 dokumenata za test
    }
]

try:
    debug_results = collection.aggregate(debug_pipeline)
    debug_results = list(debug_results)
    
    print(f"Broj dokumenata nakon osnovnog match: {len(debug_results)}")
    for i, doc in enumerate(debug_results, 1):
        print(f"{i}. {doc}")
        
except Exception as e:
    print(f"GRESKA U DEBUG MODU: {e}")

Broj dokumenata nakon osnovnog match: 5
1. {'_id': ObjectId('68f2e538bb35bab1feb700f4'), 'FlightDate': datetime.datetime(2023, 1, 2, 0, 0), 'Day_Of_Week': 1, 'Airline': 'Endeavor Air', 'Tail_Number': 'N605LR', 'Dep_Airport': 'BDL', 'Dep_CityName': 'Hartford, CT', 'DepTime_label': 'Morning', 'Dep_Delay': -3, 'Dep_Delay_Tag': 0, 'Dep_Delay_Type': 'Low <5min', 'Arr_Airport': 'LGA', 'Arr_CityName': 'New York, NY', 'Arr_Delay': -12, 'Arr_Delay_Type': 'Low <5min', 'Flight_Duration': 56, 'Distance_type': 'Short Haul >1500Mi', 'Delay_Carrier': 0, 'Delay_Weather': 0, 'Delay_NAS': 0, 'Delay_Security': 0, 'Delay_LastAircraft': 0, 'Manufacturer': 'CANADAIR REGIONAL JET', 'Model': 'CRJ', 'Aicraft_age': 16}
2. {'_id': ObjectId('68f2e538bb35bab1feb700f5'), 'FlightDate': datetime.datetime(2023, 1, 3, 0, 0), 'Day_Of_Week': 2, 'Airline': 'Endeavor Air', 'Tail_Number': 'N605LR', 'Dep_Airport': 'BDL', 'Dep_CityName': 'Hartford, CT', 'DepTime_label': 'Morning', 'Dep_Delay': -5, 'Dep_Delay_Tag': 0, 'Dep_Del

In [13]:
# Prvo testiraj lookup da vidiš da li radi
test_pipeline = [
    {
        "$match": {
            "Dep_Airport": {"$ne": None},
            "Dep_Delay": {"$ne": None}
        }
    },
    {
        "$limit": 5
    },
    {
        "$lookup": {
            "from": "airports_geolocation", 
            "localField": "Dep_Airport",
            "foreignField": "IATA_CODE",  # Probaj i sa "IATA_CODE" ako ovo ne radi
            "as": "geo_info"
        }
    }
]

try:
    test_results = collection.aggregate(test_pipeline)
    test_results = list(test_results)
    
    print("TEST REZULTATI:")
    for i, doc in enumerate(test_results, 1):
        print(f"{i}. Dep_Airport: {doc.get('Dep_Airport')}")
        print(f"   Geo info: {doc.get('geo_info')}")
        print()
        
except Exception as e:
    print(f"TEST GRESKA: {e}")

TEST REZULTATI:
1. Dep_Airport: BDL
   Geo info: [{'_id': ObjectId('68f2e8b5bb35bab1fe210cbf'), 'IATA_CODE': 'BDL', 'AIRPORT': 'Bradley International Airport', 'CITY': 'Windsor Locks', 'STATE': 'CT', 'COUNTRY': 'USA', 'LATITUDE': 41.93887, 'LONGITUDE': -72.68323}]

2. Dep_Airport: BDL
   Geo info: [{'_id': ObjectId('68f2e8b5bb35bab1fe210cbf'), 'IATA_CODE': 'BDL', 'AIRPORT': 'Bradley International Airport', 'CITY': 'Windsor Locks', 'STATE': 'CT', 'COUNTRY': 'USA', 'LATITUDE': 41.93887, 'LONGITUDE': -72.68323}]

3. Dep_Airport: BDL
   Geo info: [{'_id': ObjectId('68f2e8b5bb35bab1fe210cbf'), 'IATA_CODE': 'BDL', 'AIRPORT': 'Bradley International Airport', 'CITY': 'Windsor Locks', 'STATE': 'CT', 'COUNTRY': 'USA', 'LATITUDE': 41.93887, 'LONGITUDE': -72.68323}]

4. Dep_Airport: BDL
   Geo info: [{'_id': ObjectId('68f2e8b5bb35bab1fe210cbf'), 'IATA_CODE': 'BDL', 'AIRPORT': 'Bradley International Airport', 'CITY': 'Windsor Locks', 'STATE': 'CT', 'COUNTRY': 'USA', 'LATITUDE': 41.93887, 'LONGITUDE

**Q2:** Which airlines have the highest average delays on rainy days?

In [ ]:
base_query_2 = base_queries['query_2']

**Q3:** Which airports have the most canceled flights during bad weather?

In [ ]:
base_query_3 = base_queries['query_3']

**Q4:** Do airports with more diverse runways have lower average delays?

In [ ]:
base_query_4 = base_queries['query_4']

**Q5:** How does airline performance differ by type of flight distance (Short, Medium, Long Haul)?

In [ ]:
base_query_5 = base_queries['query_5']